# 01 — MNIST CNN Training

This notebook trains a convolutional neural network (CNN) to recognise handwritten
**Western Arabic numerals (0–9)** using the MNIST dataset.

The trained model is saved to `../models/mnist_cnn.keras` and later re-used as the
starting point (transfer learning) for the Devanagari numeral model in
`02_Devanagari_training.ipynb`.

**Pipeline overview**
1. Load and normalise the MNIST dataset
2. Build a CNN with light data augmentation and regularisation
3. Train with early stopping / LR scheduling
4. Evaluate on the held-out test set
5. Save the model, plots, and confusion matrix into `../results/`


## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import keras
from keras import layers, regularizers
from sklearn.metrics import confusion_matrix, classification_report

# Paths (relative to the notebooks/ folder)
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Keras version:", keras.__version__)


## 2. Load the dataset

We use `keras.datasets.mnist`, which downloads (or reads from the Keras cache) the
standard 60,000/10,000 train/test split.

If you'd rather train from the raw IDX files shipped in `../dataset/mnist/`
(`train-images-idx3-ubyte`, etc.), use the optional loader in the cell below instead —
just switch `USE_LOCAL_IDX_FILES` to `True`.

In [ ]:
USE_LOCAL_IDX_FILES = False

def load_idx_images(path):
    with open(path, "rb") as f:
        f.read(4)  # magic number
        n_images = int.from_bytes(f.read(4), "big")
        n_rows = int.from_bytes(f.read(4), "big")
        n_cols = int.from_bytes(f.read(4), "big")
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(n_images, n_rows, n_cols)

def load_idx_labels(path):
    with open(path, "rb") as f:
        f.read(4)  # magic number
        n_labels = int.from_bytes(f.read(4), "big")
        return np.frombuffer(f.read(), dtype=np.uint8)

if USE_LOCAL_IDX_FILES:
    MNIST_DIR = Path("../dataset/mnist")
    x_train = load_idx_images(MNIST_DIR / "train-images-idx3-ubyte")
    y_train = load_idx_labels(MNIST_DIR / "train-labels-idx1-ubyte")
    x_test = load_idx_images(MNIST_DIR / "t10k-images-idx3-ubyte")
    y_test = load_idx_labels(MNIST_DIR / "t10k-labels-idx1-ubyte")
else:
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training:", x_train.shape, y_train.shape)
print("Testing :", x_test.shape, y_test.shape)


## 3. Preprocess

Normalise pixel values to `[0, 1]` and add the channel dimension expected by `Conv2D`.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("Training:", x_train.shape)
print("Testing :", x_test.shape)


### Peek at a few samples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap="gray")
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Build the CNN

- Light `RandomRotation` / `RandomTranslation` augmentation baked into the model so it
  applies only during training.
- Two Conv → BatchNorm → MaxPool → Dropout blocks.
- A regularised dense head before the 10-way softmax output.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),

    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.1, 0.1),

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
    ),
    layers.Dropout(0.5),

    layers.Dense(10, activation="softmax"),
], name="mnist_cnn")

model.summary()


## 5. Compile

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


## 6. Callbacks

Early stopping restores the best weights (by validation loss); the LR is halved when
validation loss plateaus.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
    ),
]


## 7. Train

In [ ]:
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    callbacks=callbacks,
)


## 8. Evaluate on the test set

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


## 9. Accuracy / loss curves

Saved to `../results/accuracy_graph.png` and `../results/loss_graph.png`.

In [ ]:
hist = history.history

# Accuracy
plt.figure(figsize=(6, 4))
plt.plot(hist["accuracy"], label="train")
plt.plot(hist["val_accuracy"], label="val")
plt.title("MNIST — Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_graph.png", dpi=150)
plt.show()

# Loss
plt.figure(figsize=(6, 4))
plt.plot(hist["loss"], label="train")
plt.plot(hist["val_loss"], label="val")
plt.title("MNIST — Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_graph.png", dpi=150)
plt.show()


## 10. Confusion matrix & classification report

Saved to `../results/confusion_matrix.png`.

In [ ]:
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.title("MNIST — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(y_test, y_pred, digits=4))


## 11. Save the model

This is the checkpoint the Devanagari notebook loads for transfer learning.

In [ ]:
model.save(MODELS_DIR / "mnist_cnn.keras")
print("Saved to", MODELS_DIR / "mnist_cnn.keras")
